# Protofin AP3 — free real-model run of Agentic Causal RAG (qwen3:4b)

Runs **Causal RAG — Reduced Evidence** with genuine LLM outputs on a free GPU. Cost: **$0**.

**This notebook runs in EXPLORATORY mode.** The pre-registered reliability gates G3a and G3b are evaluated exactly as before, with the same thresholds, and their real pass/fail values are recorded in every output. The run then continues into the 95-month development replay **even if a gate fails**, so the complete pipeline can be studied end to end. That is a diagnostic, not a validation: nothing it produces is evidence that the agents are reliable or that AI adds investment value, and it can never be used for the protected historical test (`config/ap3_exploratory_mode_v1.json`).

- **Model substitution:** `qwen3:4b` (4.02B parameters, Q4_K_M, pinned digest `359d7dd4bcda`), standing in for the planned Llama 3.1 70B. The architecture, knowledge graph, point-in-time evidence, **prompts**, limits, seeds and gates are unchanged.
- **Order:** environment check → Testing Agent fixtures (G3b) → 12 sample months 2015-02..2016-01 (G3a) → the remaining development months to 2022-12.
- **Resumable:** finished months are never repeated. Results are committed after every 12 months to the private branch `ap3-qwen3-runs`. Run the notebook again to continue where it stopped.
- **Never** touches the sealed 2023–2026 window and never calls a paid service (the client accepts only a model served on this machine).

**Your one action:** Session options → Accelerator **GPU T4 x2**, Internet **on**, then **Save Version → Save & Run All (Commit)**. It runs in the background for up to 12 hours with the browser closed. If the replay has not reached 95 months, run it again; it resumes.

**One-time setup:** phone-verify the Kaggle account (needed for GPU and internet), then Add-ons → Secrets → add `GITHUB_TOKEN`, a fine-grained GitHub token for `androidilicious/protofin` with *Contents: read and write*, and attach it to this notebook.

In [ ]:
import os, subprocess, sys, json, time, pathlib
ON_KAGGLE = os.path.exists("/kaggle")
def secret(name):
    if os.environ.get(name):
        return os.environ[name]
    try:
        if ON_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None
TOKEN = secret("GITHUB_TOKEN")
assert TOKEN, "Add the GITHUB_TOKEN secret (fine-grained, androidilicious/protofin, Contents read+write) and attach it to this notebook."
os.environ["GITHUB_TOKEN"] = TOKEN
WORK = pathlib.Path("/kaggle/working" if ON_KAGGLE else "/content")
REPO = WORK / "protofin"
# git reads the token through an askpass helper that holds no secret itself; the token never goes into a clone URL
# or a .git/config, which Kaggle would save with this notebook's output
ASKPASS = pathlib.Path("/tmp/protofin-askpass.sh")
ASKPASS.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) printf "%s\\n" x-access-token ;;\n  *) printf "%s\\n" "$GITHUB_TOKEN" ;;\nesac\n')
ASKPASS.chmod(0o700)
os.environ.update(GIT_ASKPASS=str(ASKPASS), GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="credential.helper", GIT_CONFIG_VALUE_0="")
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "backtest", "https://github.com/androidilicious/protofin.git", str(REPO)], check=True)
os.chdir(REPO)
print(subprocess.run(["git", "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout)

In [ ]:
# Agent dependencies only. Kaggle and Colab already have numpy, pandas and scikit-learn; the agents do not need the
# pinned Stage 2 lock file (it targets Python 3.13). Calibration, allocation and evaluation run later, in the pinned
# environment, from the committed status features, so their numbers are reproducible exactly.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jsonschema", "pyarrow", "requests", "beautifulsoup4", "pypdf", "scikit-learn"], check=True)
print(sys.version)


In [ ]:
# Ollama: one server per GPU, two requests each, 32k context with an 8-bit KV cache
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
gpus = [l for l in subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.splitlines() if l.startswith("GPU")]
models_dir = str(WORK / "ollama_models")
urls = []
for i in range(max(1, len(gpus))):
    port = 11434 + i
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i), OLLAMA_HOST=f"127.0.0.1:{port}", OLLAMA_MODELS=models_dir,
               OLLAMA_NUM_PARALLEL="2", OLLAMA_FLASH_ATTENTION="1", OLLAMA_KV_CACHE_TYPE="q8_0", OLLAMA_KEEP_ALIVE="24h")
    subprocess.Popen(["ollama", "serve"], env=env, stdout=open(WORK / f"ollama_{i}.log", "w"), stderr=subprocess.STDOUT)
    urls.append(f"http://127.0.0.1:{port}")
time.sleep(8)
subprocess.run(["ollama", "pull", "qwen3:4b"], env=dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_MODELS=models_dir), check=True)
os.environ["AP3_OLLAMA_URLS"] = ",".join(urls)
os.environ["AP3_MONTH_WORKERS"] = str(2 * len(urls))
print(subprocess.run(["ollama", "list"], env=dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_MODELS=models_dir), capture_output=True, text=True).stdout)

In [ ]:
# Pinned inputs (SHA-256 verified) and the year-filtered knowledge-graph libraries
subprocess.run([sys.executable, "mvp/run.py", "fetch"], check=True)
subprocess.run([sys.executable, "-m", "native_backtest.scripts.build_ap3_libraries"], check=True)

In [ ]:
# check -> G3b -> G3a sample -> development replay (EXPLORATORY: continues even if a gate fails).
# Gates and thresholds are unchanged; their real values are recorded and pushed with every result.
hours = 11.0 if ON_KAGGLE else 10.0
r = subprocess.run([sys.executable, "-m", "native_backtest.scripts.ap3_free_run",
                    "--stage", "all", "--hours", str(hours), "--push", "--exploratory"])
RUN_EXIT = r.returncode     # 0 = all gates passed; 3 = a gate failed (expected here; the replay still ran); other = error
print("exit code", RUN_EXIT)

In [ ]:
# What exists now (the same files are on branch ap3-qwen3-runs)
s = json.loads((REPO / "native_backtest/out/ap3/qwen3_runs/summary.json").read_text())
print(json.dumps({k: s[k] for k in ("n_months", "target_months", "cost_usd")}, indent=1))
print(json.dumps(s.get("protocol"), indent=1))
print(json.dumps({g: v.get("pass") for g, v in s["gates"].items()}, indent=1))
x = s.get("exploratory")
print("EXPLORATORY:", x["label"] if x else "no (gates passed)")
if x:
    print(json.dumps(x["gates_failed"], indent=1))
print(json.dumps(s.get("tokens_this_session"), indent=1))
rel = s.get("reliability_this_session") or {}
print(json.dumps({k: rel.get(k) for k in ("months_with_hypotheses", "months_final_answer_forced", "repeated_tool_calls",
                                          "hypotheses", "tests", "failed_tests", "status_counts")}, indent=1, default=str))

for e in s.get("reliability_earlier_sessions") or []:
    print("earlier session", e.get("written_utc"), f'({e.get("n_months")} months, gates {e.get("gates")})',
          json.dumps({k: (e.get("report") or {}).get(k) for k in ("hypotheses", "tests", "failed_tests")}, default=str))

done, need = s["n_months"], s["target_months"]
print(f"\n{done} of {need} development months complete.")
if done < need:
    print("Not finished: run this notebook again (Save Version -> Save & Run All). It resumes and skips finished months.")
else:
    print("All months complete. Tell Claude, who imports from branch ap3-qwen3-runs and runs the backtest.")
# An error (anything other than 0 or 3) must not look like a finished run.
assert RUN_EXIT in (0, 3), f"ap3_free_run failed with exit code {RUN_EXIT}: see the run cell output"